In [5]:
# ================================================================
#  Energy-based Out-of-Distribution Detection
#  CIFAR-10 (In-Distribution)  vs  SVHN (OOD)
#  Tập trung: Quét ngưỡng (threshold sweep) và tìm ngưỡng tối ưu
#
#  Dựa trên: Liu et al., "Energy-based OOD Detection", NeurIPS 2020
#  Chạy trên: Google Colab (free tier, GPU Tesla T4)
# ================================================================

# ── Cài đặt thư viện ────────────────────────────────────────────
# !pip install torch torchvision scikit-learn matplotlib tqdm -q

# ================================================================
# SECTION 1 – Imports & cấu hình
# ================================================================
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (roc_curve, auc,
                             precision_recall_curve,
                             average_precision_score)
from tqdm import tqdm

# ── Reproducibility ─────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = 'wrn28_10_cifar10.pth'
EPOCHS     = 20          # ~1.5 h trên Colab free tier T4
BATCH      = 128
T          = 1.0          # temperature energy score

print(f"Device: {DEVICE}")



Device: cuda


In [6]:

# ================================================================
# SECTION 2 – WideResNet-28-10
# ================================================================
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride, drop=0.0):
        super().__init__()
        self.bn1  = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn2  = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.drop  = drop
        self.skip  = nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False) \
                     if in_ch != out_ch else None

    def forward(self, x):
        out = F.relu(self.bn1(x), inplace=True)
        sc  = self.skip(out) if self.skip else x
        out = self.conv1(out)
        out = F.relu(self.bn2(out), inplace=True)
        if self.drop > 0:
            out = F.dropout(out, p=self.drop, training=self.training)
        return self.conv2(out) + sc


class NetBlock(nn.Module):
    def __init__(self, n, in_ch, out_ch, stride, drop=0.0):
        super().__init__()
        self.net = nn.Sequential(*[
            BasicBlock(in_ch if i == 0 else out_ch,
                       out_ch, stride if i == 0 else 1, drop)
            for i in range(n)
        ])
    def forward(self, x): return self.net(x)


class WideResNet(nn.Module):
    def __init__(self, depth=28, k=10, num_classes=10, drop=0.0):
        super().__init__()
        assert (depth - 4) % 6 == 0
        n  = (depth - 4) // 6
        ch = [16, 16*k, 32*k, 64*k]
        self.conv0  = nn.Conv2d(3, ch[0], 3, padding=1, bias=False)
        self.block1 = NetBlock(n, ch[0], ch[1], 1, drop)
        self.block2 = NetBlock(n, ch[1], ch[2], 2, drop)
        self.block3 = NetBlock(n, ch[2], ch[3], 2, drop)
        self.bn     = nn.BatchNorm2d(ch[3])
        self.fc     = nn.Linear(ch[3], num_classes)
        self.out_ch = ch[3]
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1); m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                m.bias.data.zero_()

    def forward(self, x):
        x = self.conv0(x)
        x = self.block1(x); x = self.block2(x); x = self.block3(x)
        x = F.relu(self.bn(x), inplace=True)
        x = F.adaptive_avg_pool2d(x, 1).view(-1, self.out_ch)
        return self.fc(x)



In [7]:
# ================================================================
# SECTION 3 – Data
# ================================================================
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

cifar_train = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=train_tf)
cifar_test  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=test_tf)
svhn_test   = torchvision.datasets.SVHN(  './data', split='test', download=True, transform=test_tf)

train_ldr  = DataLoader(cifar_train, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
id_ldr     = DataLoader(cifar_test,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
ood_ldr    = DataLoader(svhn_test,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f"CIFAR-10 train={len(cifar_train):,}  test={len(cifar_test):,}")
print(f"SVHN test={len(svhn_test):,}")



CIFAR-10 train=50,000  test=10,000
SVHN test=26,032


In [8]:

# ================================================================
# SECTION 4 – Train / Load model
# ================================================================
def train(model, loader, epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1,
                          momentum=0.9, weight_decay=5e-4, nesterov=True)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    model.train()
    for ep in range(1, epochs + 1):
        loss_sum = correct = total = 0
        for imgs, labels in tqdm(loader, desc=f"Epoch {ep:3d}/{epochs}", leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward(); optimizer.step()
            loss_sum += loss.item() * len(imgs)
            correct  += (out.argmax(1) == labels).sum().item()
            total    += len(imgs)
        scheduler.step()
        if ep % 20 == 0:
            print(f"  ep{ep}: loss={loss_sum/total:.4f}  acc={100*correct/total:.2f}%")
    return model

model = WideResNet(depth=28, k=10).to(DEVICE)

if os.path.exists(CHECKPOINT):
    print(f"Loading checkpoint: {CHECKPOINT}")
    model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
else:
    print("Training WideResNet-28-10 (~1.5 h on Colab free tier)…")
    model = train(model, train_ldr, EPOCHS)
    torch.save(model.state_dict(), CHECKPOINT)
    print(f"Saved → {CHECKPOINT}")

# ── Accuracy check ──────────────────────────────────────────────
model.eval()
ok = tot = 0
with torch.no_grad():
    for x, y in id_ldr:
        p = model(x.to(DEVICE)).argmax(1)
        ok  += (p == y.to(DEVICE)).sum().item()
        tot += len(y)
print(f"CIFAR-10 test accuracy: {100*ok/tot:.2f}%")



Training WideResNet-28-10 (~1.5 h on Colab free tier)…


  ep20: loss=0.0242  acc=99.35%
Saved → wrn28_10_cifar10.pth
CIFAR-10 test accuracy: 93.84%


In [ ]:
# ================================================================
# SECTION 5 – Scoring functions
# ================================================================
def get_scores(model, loader, mode='energy', temperature=1.0):
    """
    mode='energy' → E(x) = -T * log Σ exp(f_k / T)   (high = OOD)
    mode='msp'    → -max softmax(f)                   (high = OOD)
    """
    model.eval()
    out = []
    with torch.no_grad():
        for imgs, _ in loader:
            logits = model(imgs.to(DEVICE))
            if mode == 'energy':
                s = -temperature * torch.logsumexp(logits / temperature, dim=1)
            else:                       # MSP: negate so high → OOD
                s = -F.softmax(logits, dim=1).max(dim=1).values
            out.append(s.cpu().numpy())
    return np.concatenate(out)

print("Computing scores…")
e_id  = get_scores(model, id_ldr,  mode='energy', temperature=T)
e_ood = get_scores(model, ood_ldr, mode='energy', temperature=T)
m_id  = get_scores(model, id_ldr,  mode='msp')
m_ood = get_scores(model, ood_ldr, mode='msp')

print(f"Energy  ID  mean={e_id.mean():.3f}  std={e_id.std():.3f}")
print(f"Energy  OOD mean={e_ood.mean():.3f}  std={e_ood.std():.3f}")
print(f"MSP     ID  mean={m_id.mean():.3f}  std={m_id.std():.3f}")
print(f"MSP     OOD mean={m_ood.mean():.3f}  std={m_ood.std():.3f}")



In [ ]:

# ================================================================
# SECTION 6 – Threshold sweep & metrics
# ================================================================
def sweep(scores_id, scores_ood, n=2000):
    """
    Với mỗi ngưỡng γ:
      - mẫu được gắn nhãn OOD (=1) nếu score >= γ
      - tính FPR, TPR, Precision, Recall, F1, Youden's J
    """
    all_scores = np.concatenate([scores_id, scores_ood])
    all_labels = np.concatenate([np.zeros(len(scores_id)),
                                 np.ones(len(scores_ood))])
    thresholds = np.linspace(all_scores.min(), all_scores.max(), n)

    fprs = np.empty(n); tprs = np.empty(n)
    precs = np.empty(n); f1s = np.empty(n)

    for i, γ in enumerate(thresholds):
        pred = (all_scores >= γ).astype(int)
        TP = ((pred == 1) & (all_labels == 1)).sum()
        FP = ((pred == 1) & (all_labels == 0)).sum()
        TN = ((pred == 0) & (all_labels == 0)).sum()
        FN = ((pred == 0) & (all_labels == 1)).sum()

        tprs[i]  = TP / (TP + FN + 1e-9)
        fprs[i]  = FP / (FP + TN + 1e-9)
        precs[i] = TP / (TP + FP + 1e-9)
        f1s[i]   = 2*precs[i]*tprs[i] / (precs[i] + tprs[i] + 1e-9)

    return thresholds, fprs, tprs, precs, f1s


def fpr_at_tpr(scores_id, scores_ood, tpr_target=0.95):
    """Ngưỡng chuẩn của bài báo: tìm γ sao cho TPR = tpr_target."""
    labels = np.concatenate([np.zeros(len(scores_id)), np.ones(len(scores_ood))])
    scores = np.concatenate([scores_id, scores_ood])
    fpr_v, tpr_v, thr_v = roc_curve(labels, scores)
    idx = np.argmin(np.abs(tpr_v - tpr_target))
    return thr_v[idx], fpr_v[idx], tpr_v[idx]


print("\nSweeping thresholds…")
th_e, fpr_e, tpr_e, prec_e, f1_e = sweep(e_id, e_ood)
th_m, fpr_m, tpr_m, prec_m, f1_m = sweep(m_id, m_ood)

# ── Three criteria for best threshold ───────────────────────────
γ_fpr95_e, fpr95_e, _  = fpr_at_tpr(e_id, e_ood)   # (1) paper standard
γ_fpr95_m, fpr95_m, _  = fpr_at_tpr(m_id, m_ood)

γ_f1_e = th_e[np.argmax(f1_e)]                      # (2) max F1
γ_f1_m = th_m[np.argmax(f1_m)]

youden_e = tpr_e - fpr_e
γ_youden_e = th_e[np.argmax(youden_e)]              # (3) Youden's J

# ── ROC / PR ────────────────────────────────────────────────────
lbl_e = np.concatenate([np.zeros(len(e_id)), np.ones(len(e_ood))])
sc_e  = np.concatenate([e_id, e_ood])
lbl_m = np.concatenate([np.zeros(len(m_id)), np.ones(len(m_ood))])
sc_m  = np.concatenate([m_id, m_ood])

fpr_roc_e, tpr_roc_e, _ = roc_curve(lbl_e, sc_e)
fpr_roc_m, tpr_roc_m, _ = roc_curve(lbl_m, sc_m)
auroc_e = auc(fpr_roc_e, tpr_roc_e)
auroc_m = auc(fpr_roc_m, tpr_roc_m)

prec_pr_e, rec_pr_e, _ = precision_recall_curve(lbl_e, sc_e)
prec_pr_m, rec_pr_m, _ = precision_recall_curve(lbl_m, sc_m)
ap_e = average_precision_score(lbl_e, sc_e)
ap_m = average_precision_score(lbl_m, sc_m)



In [ ]:

# ================================================================
# SECTION 7 – Visualization (6 biểu đồ)
# ================================================================
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

BLUE   = '#1976D2'
RED    = '#D32F2F'
ORANGE = '#F57C00'
GREEN  = '#388E3C'
PURPLE = '#7B1FA2'
TEAL   = '#00796B'

fig = plt.figure(figsize=(20, 15))
fig.suptitle(
    'Energy-based OOD Detection — Phân Tích Ngưỡng (Threshold Analysis)\n'
    'CIFAR-10 (In-Distribution)  vs  SVHN (Out-of-Distribution)',
    fontsize=15, fontweight='bold', y=1.005
)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.48, wspace=0.33)


# ── [0,0:2]  Biểu đồ 1: Phân phối Energy Score ─────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.hist(e_id,  bins=100, alpha=0.60, color=BLUE,  density=True,
         label='CIFAR-10 – In-Distribution')
ax1.hist(e_ood, bins=100, alpha=0.60, color=RED,   density=True,
         label='SVHN – Out-of-Distribution')
ax1.axvline(γ_fpr95_e,  color=GREEN,  lw=2.0, ls='--',
            label=f'γ (TPR=95%) = {γ_fpr95_e:.2f}')
ax1.axvline(γ_f1_e,     color=PURPLE, lw=2.0, ls=':',
            label=f'γ (Best F1) = {γ_f1_e:.2f}')
ax1.axvline(γ_youden_e, color=TEAL,   lw=2.0, ls='-.',
            label=f'γ (Youden) = {γ_youden_e:.2f}')
ax1.set_xlabel('Energy Score  E(x)', fontsize=11)
ax1.set_ylabel('Mật độ xác suất', fontsize=11)
ax1.set_title('① Phân phối Energy Score — In-Dist vs OOD\n'
              '(Khoảng tách biệt càng lớn → phát hiện OOD càng dễ)',
              fontsize=11, fontweight='bold')
ax1.legend(fontsize=8.5)


# ── [0,2]  Biểu đồ 2: ROC Curve ────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2.plot(fpr_roc_e, tpr_roc_e, color=BLUE,   lw=2,
         label=f'Energy  AUROC={auroc_e:.3f}')
ax2.plot(fpr_roc_m, tpr_roc_m, color=ORANGE, lw=2, ls='--',
         label=f'MSP     AUROC={auroc_m:.3f}')
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.35, label='Random (0.500)')
ax2.scatter([fpr95_e], [0.95], color=GREEN, s=90, zorder=6,
            label=f'FPR95={fpr95_e:.3f}')
ax2.scatter([fpr95_m], [0.95], color=ORANGE, s=90, marker='^', zorder=6)
ax2.set_xlabel('False Positive Rate', fontsize=11)
ax2.set_ylabel('True Positive Rate', fontsize=11)
ax2.set_title('② ROC Curve\n(diện tích lớn hơn = tổng thể tốt hơn)',
              fontsize=11, fontweight='bold')
ax2.legend(fontsize=8.5)


# ── [1,0:2]  Biểu đồ 3: FPR & TPR vs Threshold (CHÍNH) ─────────
ax3 = fig.add_subplot(gs[1, :2])
ax3.plot(th_e, fpr_e, color=RED,   lw=2.2,
         label='FPR — tỷ lệ OOD bị bỏ sót (↓ muốn thấp)')
ax3.plot(th_e, tpr_e, color=GREEN, lw=2.2,
         label='TPR — tỷ lệ OOD phát hiện đúng (↑ muốn cao)')
ax3.axhline(0.95, color='gray', lw=1, ls=':', alpha=0.6)
ax3.text(th_e.max() * 0.99, 0.96, 'TPR = 95%', ha='right',
         fontsize=8, color='gray')

# Vùng đặt ngưỡng
ax3.axvline(γ_fpr95_e, color=GREEN,  lw=2, ls='--',
            label=f'① γ = {γ_fpr95_e:.2f}  (TPR=95%, FPR={fpr95_e:.3f})')
ax3.axvline(γ_f1_e,    color=PURPLE, lw=2, ls=':',
            label=f'② γ = {γ_f1_e:.2f}  (Best F1={f1_e.max():.3f})')
ax3.axvline(γ_youden_e,color=TEAL,   lw=2, ls='-.',
            label=f'③ γ = {γ_youden_e:.2f}  (Youden J={youden_e.max():.3f})')

# Tô màu vùng ngưỡng
ax3.fill_between(th_e, 0, 1,
                 where=(th_e >= γ_fpr95_e - 0.5) & (th_e <= γ_fpr95_e + 0.5),
                 alpha=0.10, color=GREEN, label='Vùng ngưỡng tối ưu (FPR95)')

ax3.set_xlabel('Ngưỡng  γ  (Energy Score)', fontsize=11)
ax3.set_ylabel('Tỷ lệ', fontsize=11)
ax3.set_title('③ FPR & TPR theo Ngưỡng γ — Tìm điểm cân bằng tối ưu\n'
              '(Ngưỡng nhỏ → nhiều False Alarm | Ngưỡng lớn → bỏ sót OOD)',
              fontsize=11, fontweight='bold')
ax3.legend(fontsize=8.5, loc='center right')
ax3.set_xlim(th_e.min(), th_e.max())
ax3.set_ylim(-0.02, 1.05)


# ── [1,2]  Biểu đồ 4: F1 & Youden's J vs Threshold ─────────────
ax4 = fig.add_subplot(gs[1, 2])
ax4.plot(th_e, f1_e,     color=PURPLE, lw=2,
         label=f'F1  (max={f1_e.max():.3f})')
ax4.plot(th_e, youden_e, color=TEAL,   lw=2, ls='--',
         label=f"Youden J (max={youden_e.max():.3f})")
ax4.axvline(γ_f1_e,    color=PURPLE, lw=1.5, ls=':',
            label=f'Best F1 → γ={γ_f1_e:.2f}')
ax4.axvline(γ_youden_e,color=TEAL,   lw=1.5, ls='-.',
            label=f'Best J  → γ={γ_youden_e:.2f}')
ax4.set_xlabel('Ngưỡng  γ', fontsize=11)
ax4.set_ylabel('Score', fontsize=11)
ax4.set_title('④ F1 & Youden\'s J theo Ngưỡng\n'
              '(Đỉnh = ngưỡng tối ưu theo tiêu chí đó)',
              fontsize=11, fontweight='bold')
ax4.legend(fontsize=8.5)


# ── [2,0]  Biểu đồ 5: Precision-Recall Curve ───────────────────
ax5 = fig.add_subplot(gs[2, 0])
ax5.plot(rec_pr_e, prec_pr_e, color=BLUE,   lw=2,
         label=f'Energy  AP={ap_e:.3f}')
ax5.plot(rec_pr_m, prec_pr_m, color=ORANGE, lw=2, ls='--',
         label=f'MSP     AP={ap_m:.3f}')
ax5.set_xlabel('Recall', fontsize=11)
ax5.set_ylabel('Precision', fontsize=11)
ax5.set_title('⑤ Precision-Recall Curve\n(diện tích lớn = phát hiện OOD cân bằng hơn)',
              fontsize=11, fontweight='bold')
ax5.legend(fontsize=9)


# ── [2,1]  Biểu đồ 6: So sánh tổng hợp (bar chart) ─────────────
ax6 = fig.add_subplot(gs[2, 1])
metric_names = ['FPR95 (%)\n↓ thấp hơn tốt hơn',
                'AUROC (%)\n↑ cao hơn tốt hơn',
                'Best F1 (%)\n↑ cao hơn tốt hơn']
e_vals = [fpr95_e*100, auroc_e*100, f1_e.max()*100]
m_vals = [fpr95_m*100, auroc_m*100, f1_m.max()*100]
x = np.arange(len(metric_names))
w = 0.35
b1 = ax6.bar(x - w/2, e_vals, w, color=BLUE,   alpha=0.85, label='Energy Score')
b2 = ax6.bar(x + w/2, m_vals, w, color=ORANGE, alpha=0.85, label='MSP Baseline')
for b, lbl_val in zip(list(b1)+list(b2), e_vals+m_vals):
    ax6.text(b.get_x() + b.get_width()/2,
             b.get_height() + 0.8,
             f'{lbl_val:.1f}', ha='center', va='bottom',
             fontsize=8.5, fontweight='bold')
ax6.set_xticks(x); ax6.set_xticklabels(metric_names, fontsize=8.5)
ax6.set_ylim(0, 115)
ax6.set_title('⑥ So sánh tổng hợp Energy vs MSP\n(cùng mô hình, không train lại)',
              fontsize=11, fontweight='bold')
ax6.legend(fontsize=9)


# ── [2,2]  Biểu đồ 7: Bảng tóm tắt ngưỡng ──────────────────────
ax7 = fig.add_subplot(gs[2, 2])
ax7.axis('off')
rows = [
    ['Tiêu chí chọn γ', 'Ngưỡng γ', 'FPR', 'TPR'],
    ['① TPR = 95%\n(chuẩn bài báo)',
     f'{γ_fpr95_e:.3f}',
     f'{fpr95_e:.3f}',
     '0.950'],
    ['② Max F1\n(cân bằng P/R)',
     f'{γ_f1_e:.3f}',
     f'{fpr_e[np.argmax(f1_e)]:.3f}',
     f'{tpr_e[np.argmax(f1_e)]:.3f}'],
    ['③ Youden\'s J\n(max TPR-FPR)',
     f'{γ_youden_e:.3f}',
     f'{fpr_e[np.argmax(youden_e)]:.3f}',
     f'{tpr_e[np.argmax(youden_e)]:.3f}'],
]
col_colors = [[BLUE]*4]
cell_colors = [
    ['#E3F2FD']*4,
    ['#E8F5E9']*4,
    ['#EDE7F6']*4,
]
tbl = ax7.table(
    cellText=rows[1:], colLabels=rows[0],
    cellLoc='center', loc='center',
    cellColours=cell_colors,
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.1, 2.0)
ax7.set_title('⑦ Tóm tắt 3 tiêu chí\nchọn ngưỡng tối ưu',
              fontsize=11, fontweight='bold', pad=12)


plt.savefig('ood_threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Đã lưu: ood_threshold_analysis.png")




In [ ]:
# ================================================================
# SECTION 8 – In kết quả tổng hợp
# ================================================================
sep = '=' * 62
print(f'\n{sep}')
print('   KẾT QUẢ PHÂN TÍCH NGƯỠNG — ENERGY OOD DETECTION')
print(f'{sep}')
print(f'{"Phương pháp":>22} | {"FPR95":>7} | {"AUROC":>7} | {"Best F1":>8}')
print('-'*55)
print(f'{"Energy Score":>22} | {fpr95_e*100:>6.2f}% | {auroc_e*100:>6.2f}% | {f1_e.max():>8.4f}')
print(f'{"MSP (Baseline)":>22} | {fpr95_m*100:>6.2f}% | {auroc_m*100:>6.2f}% | {f1_m.max():>8.4f}')
print(f'{sep}')
print('\n📌 Ngưỡng khuyến nghị cho Energy Score:')
print(f'   ① γ = {γ_fpr95_e:.4f}  →  tiêu chuẩn FPR95 (bài báo gốc)')
print(f'      FPR = {fpr95_e*100:.2f}%  |  TPR = 95.00%')
print(f'   ② γ = {γ_f1_e:.4f}   →  cân bằng Precision-Recall (Max F1)')
print(f'      F1  = {f1_e.max():.4f}')
print(f'   ③ γ = {γ_youden_e:.4f}   →  tối ưu Youden\'s J (Max TPR-FPR)')
print(f'      J   = {youden_e.max():.4f}')
print(f'\n💡 Khuyến nghị triển khai:')
print(f'   Dùng γ = {γ_fpr95_e:.4f} để tương thích với benchmark bài báo.')
print(f'   Nếu ưu tiên ít báo nhầm hơn → tăng γ (FPR giảm, nhưng TPR giảm).')
print(f'   Nếu ưu tiên bắt nhiều OOD hơn → giảm γ (TPR tăng, FPR tăng).\n')